# Run BioEmu In Colab

Thin Colab wrapper for generating a portable BioEmu bundle with `manifest.json` and `nodes/*.pdb`.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Install BioEmu And This Package

Set `REPO_URL` to this repository's Git URL if the clone cell needs to fetch it into Colab.

In [ ]:
%pip install -q "bioemu[cuda]"

from pathlib import Path

REPO_DIR = Path("/content/AA_Protein-Conformation-Guided-Random-Walk")
REPO_URL = ""  # Example: "https://github.com/your-org/AA_Protein-Conformation-Guided-Random-Walk.git"

if not REPO_DIR.exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL before cloning this repository in Colab.")
    !git clone {REPO_URL} {REPO_DIR}

%pip install -q -e {REPO_DIR}

## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Configure Input And Output Paths

Upload or place a FASTA file in Drive, then adjust `FASTA_PATH` and `OUTPUT_DIR`.

In [ ]:
from pathlib import Path

FASTA_PATH = Path("/content/drive/MyDrive/bioemu_inputs/protein.fasta")
OUTPUT_DIR = Path("/content/drive/MyDrive/bioemu_outputs/protein_bundle")
N_SAMPLES = 200
SEED = 7

assert FASTA_PATH.exists(), f"Missing FASTA: {FASTA_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FASTA:", FASTA_PATH)
print("Output:", OUTPUT_DIR)

## 5. Run BioEmu And Write Portable Bundle

If your installed BioEmu version exposes a different API, pass a custom `sampler=` callable to `run_bioemu_from_fasta`. The callable can write PDB files into `output_dir` or return PDB paths, PDB text, or coordinate arrays.

In [ ]:
from guided_conformational_path.bioemu_runner import run_bioemu_from_fasta

manifest = run_bioemu_from_fasta(
    fasta_path=FASTA_PATH,
    output_dir=OUTPUT_DIR,
    n_samples=N_SAMPLES,
    seed=SEED,
)

print("Wrote", manifest["run"]["n_samples_written"], "PDB nodes")
print("Manifest:", OUTPUT_DIR / "manifest.json")
print("Nodes:", OUTPUT_DIR / "nodes")